# 14 · Statistical Validation

**Goal.** Show the results are **stable and statistically real**, not a lucky run.

**Why it matters.** Reporting a single number can mislead. We retrain with several random seeds, report
**confidence intervals**, and run **significance tests** against simple baselines.

**What happens below.** Multi-seed training, confidence intervals, paired tests with multiple-comparison
correction, and cross-validation.

**Output:** `data/stage5_validation/seed_metrics.json` + `significance.json`.

## 0. Environment & dependencies

In [1]:
import os, sys, json, time, math, logging, warnings, random
from pathlib import Path
from collections import Counter

for _stream in ("stdout", "stderr"):
    try:
        getattr(sys, _stream).reconfigure(encoding="utf-8")
    except Exception:
        pass

import numpy as np
def _try(name):
    try:
        __import__(name); return True
    except Exception:
        return False
assert _try("torch") and _try("sklearn"), "notebook 14 needs torch + sklearn"
HAS_SCIPY = _try("scipy")
HAS_MPL = _try("matplotlib")

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("stats")
PART_VERSION = "1.0"
SEEDS = [42, 2023, 2024, 0, 7]   # D1
log.info("seeds = %s | scipy=%s", SEEDS, HAS_SCIPY)

2026-08-08 20:26:54,408 | INFO | seeds = [42, 2023, 2024, 0, 7] | scipy=True


## 1. Config & paths (reuse the Objective 3 hyperparameters verbatim)

In [2]:
import torch, torch.nn as nn, torch.nn.functional as F

def find_project_root():
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "data" / "stage3_model").exists() or (d / "data" / "stage2_semantic").exists() or (d / ".env").exists():
            return d
    return here

PROJECT_ROOT = find_project_root()
SEM_DIR  = PROJECT_ROOT / "data" / "stage2_semantic"
TEMP_DIR = PROJECT_ROOT / "data" / "stage1_temporal"
MODEL_DIR = PROJECT_ROOT / "data" / "stage3_model"
OUT_DIR = PROJECT_ROOT / "data" / "stage5_validation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CFG = json.loads((MODEL_DIR / "model_config.json").read_text(encoding="utf-8"))
FUSION_DIM = CFG["fusion_dim"]; PRIM_CAPS = CFG["prim_caps"]; PRIM_DIM = CFG["prim_dim"]
CLASS_DIM = CFG["class_dim"]; N_CLASSES = 4
ROUTING = CFG["routing"]; GATE = CFG["gate"]; DROPOUT = 0.3
TEMP_FEATS = CFG["temp_feats"]
EPOCHS = CFG["epochs"]; LR = CFG["lr"]; BATCH = CFG["batch"]
FOCAL_GAMMA = CFG["focal_gamma"]; CONTRAST_W = CFG["contrast_w"]; CONTRAST_T = 0.1
WEIGHT_DECAY = 1e-4
DEVICE = "cpu"
TIER_NAMES = {0: "Neutral", 1: "Stress", 2: "Moderate", 3: "Crisis"}
log.info("hparams: epochs=%d lr=%.0e batch=%d routing=%s gate=%s focal_gamma=%.1f contrast_w=%.2f",
         EPOCHS, LR, BATCH, ROUTING, GATE, FOCAL_GAMMA, CONTRAST_W)

2026-08-08 20:26:54,420 | INFO | hparams: epochs=60 lr=1e-03 batch=16 routing=static gate=elu focal_gamma=2.0 contrast_w=0.20


## 2. Model + losses (verbatim from notebook 09, so the multi-seed model is the SAME network)

In [3]:
def squash(s, dim=-1, eps=1e-8):
    n2 = (s * s).sum(dim=dim, keepdim=True)
    return (n2 / (1.0 + n2)) * (s / torch.sqrt(n2 + eps))

class CapsuleRouting(nn.Module):
    def __init__(self, n_in, d_in, n_out, d_out, routing="static", iters=3):
        super().__init__()
        self.n_in, self.n_out, self.routing, self.iters = n_in, n_out, routing, iters
        self.W = nn.Parameter(0.01 * torch.randn(1, n_in, n_out, d_out, d_in))
        self.b_static = nn.Parameter(torch.zeros(1, n_in, n_out))
    def forward(self, u):
        B = u.size(0)
        u_hat = torch.matmul(self.W, u.unsqueeze(2).unsqueeze(-1)).squeeze(-1)
        if self.routing == "static":
            c = torch.softmax(self.b_static, dim=2).unsqueeze(-1)
            return squash((c * u_hat).sum(dim=1))
        b = torch.zeros(B, self.n_in, self.n_out, device=u.device)
        for it in range(self.iters):
            c = torch.softmax(b, dim=2).unsqueeze(-1)
            v = squash((c * u_hat).sum(dim=1))
            if it < self.iters - 1:
                b = b + (u_hat * v.unsqueeze(1)).sum(-1)
        return v

class DriftCapsNet(nn.Module):
    def __init__(self, text_dim, emo_dim, temp_dim=6, d=FUSION_DIM, gate=GATE, fusion="attention",
                 routing=ROUTING, dropout=DROPOUT):
        super().__init__()
        self.gate_kind, self.fusion = gate, fusion
        self.text_proj = nn.Linear(text_dim, PRIM_CAPS * PRIM_DIM)
        self.gate_lin  = nn.Linear(PRIM_CAPS * PRIM_DIM, PRIM_CAPS * PRIM_DIM)
        self.caps = CapsuleRouting(PRIM_CAPS, PRIM_DIM, N_CLASSES, CLASS_DIM, routing=routing)
        self.q_proj = nn.Linear(N_CLASSES * CLASS_DIM, d)
        self.e_proj = nn.Linear(emo_dim, d)
        self.t_proj = nn.Sequential(nn.Linear(temp_dim, 64), nn.ReLU(), nn.Linear(64, d))
        self.drop = nn.Dropout(dropout); self.norm = nn.LayerNorm(d)
        head_in = d if fusion in ("attention", "text_only") else 3 * d
        self.head = nn.Linear(head_in, N_CLASSES); self.d = d
    def gate(self, H):
        if self.gate_kind == "none":    return H
        if self.gate_kind == "sigmoid": return H * torch.sigmoid(self.gate_lin(H))
        return H * (F.elu(self.gate_lin(H)) + 1.0)
    def temporal_rep(self, x_t, mask):
        return self.t_proj(x_t) * mask.unsqueeze(-1)
    def forward(self, x_text, x_emo, x_temp, mask, return_z=False):
        H = self.drop(F.elu(self.text_proj(x_text)))
        U = squash(self.gate(H).view(-1, PRIM_CAPS, PRIM_DIM))
        V = self.caps(U).reshape(x_text.size(0), -1)
        q = self.q_proj(V); e = self.e_proj(x_emo); tau = self.temporal_rep(x_temp, mask)
        if self.fusion == "attention":
            kv = torch.stack([tau, e], dim=1)
            att = torch.softmax(torch.matmul(q.unsqueeze(1), kv.transpose(1, 2)) / math.sqrt(self.d), dim=-1)
            z = self.norm(q + torch.matmul(att, kv).squeeze(1))
        elif self.fusion == "concat":
            z = torch.cat([q, tau, e], dim=-1)
        else:
            z = q
        logits = self.head(z)
        return (logits, F.normalize(z, dim=-1)) if return_z else logits

class FocalLoss(nn.Module):
    def __init__(self, weight, gamma=FOCAL_GAMMA):
        super().__init__(); self.weight = weight; self.gamma = gamma
    def forward(self, logits, y):
        logp = F.log_softmax(logits, dim=-1); p = logp.exp(); w = self.weight[y]
        pt = p.gather(1, y.unsqueeze(1)).squeeze(1).clamp(1e-6, 1.0)
        logpt = logp.gather(1, y.unsqueeze(1)).squeeze(1)
        return (-w * (1 - pt) ** self.gamma * logpt).mean()

def supervised_contrastive(z, y, temp=CONTRAST_T):
    sim = torch.matmul(z, z.t()) / temp; B = z.size(0)
    lm = ~torch.eye(B, dtype=torch.bool, device=z.device)
    pos = (y.unsqueeze(0) == y.unsqueeze(1)) & lm
    sim = sim.masked_fill(~lm, -1e9)
    logp = sim - torch.logsumexp(sim, dim=1, keepdim=True)
    pc = pos.sum(1); valid = pc > 0
    if valid.sum() == 0:
        return torch.tensor(0.0, device=z.device)
    return -((logp.masked_fill(~pos, 0.0).sum(1)[valid]) / pc[valid].clamp(min=1)).mean()

log.info("model + losses defined (verbatim notebook 09)")

2026-08-08 20:26:54,439 | INFO | model + losses defined (verbatim notebook 09)


## 3. Load cached frozen embeddings + temporal features (no model reload — fast)

In [4]:
def load_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]

train_rows = load_jsonl(SEM_DIR / "corpus_preprocessed_train.jsonl")
test_rows  = load_jsonl(SEM_DIR / "corpus_preprocessed_test.jsonl")
assert set(r["platform"] for r in train_rows) <= {"youtube", "synthetic"}, "TRAIN platform leak!"
assert set(r["platform"] for r in test_rows)  <= {"reddit", "twitter"},   "TEST platform leak!"

def muril_matrix(split, rows):
    emb = np.load(SEM_DIR / f"embeddings_{split}.npy").astype(np.float32)
    idx = json.loads((SEM_DIR / f"embeddings_index_{split}.json").read_text(encoding="utf-8"))
    id2row = {rid: i for i, rid in enumerate(idx)}
    return np.vstack([emb[id2row[r["record_id"]]] for r in rows])

MURIL_TR = muril_matrix("train", train_rows); MURIL_TE = muril_matrix("test", test_rows)
EMO_TR = np.load(MODEL_DIR / "emo_embeddings_train.npy").astype(np.float32)
EMO_TE = np.load(MODEL_DIR / "emo_embeddings_test.npy").astype(np.float32)
assert EMO_TR.shape[0] == len(train_rows) and EMO_TE.shape[0] == len(test_rows), "emo cache misaligned!"

def load_temporal(split):
    return {r["record_id"]: r for r in load_jsonl(TEMP_DIR / f"temporal_features_{split}.jsonl")}
temp_tr, temp_te = load_temporal("train"), load_temporal("test")

def temporal_matrix(rows, by_id):
    feats, mask = [], []
    for r in rows:
        t = by_id.get(r["record_id"], {})
        feats.append([float(t.get(k, 0.0)) for k in TEMP_FEATS])
        mask.append(float(t.get("temporal_mask", r.get("temporal_mask", 1))))
    return np.array(feats, np.float32), np.array(mask, np.float32)

TEMP_TR, MASK_TR = temporal_matrix(train_rows, temp_tr)
TEMP_TE, MASK_TE = temporal_matrix(test_rows, temp_te)
y_train = np.array([int(r["tier"]) for r in train_rows])
y_test  = np.array([int(r["tier"]) for r in test_rows])
log.info("train=%d test=%d | MuRIL=%s EMO=%s | TEST scored once per seed",
         len(train_rows), len(test_rows), MURIL_TR.shape, EMO_TR.shape)

2026-08-08 20:26:54,756 | INFO | train=1971 test=2768 | MuRIL=(1971, 768) EMO=(1971, 768) | TEST scored once per seed


## 4. Training + metric drivers (seeded; backbones frozen, D2)

`train_one(...)` trains the head (full fusion or text-only) on a given TRAIN subset under a seed; `evaluate(...)`
returns the metric suite. Seeds control init + batch shuffle, so cross-seed spread = the model's stability.

In [5]:
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (accuracy_score, f1_score, matthews_corrcoef, roc_auc_score,
                             recall_score)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

def to_t(*a): return [torch.tensor(x) for x in a]

def metric_suite(y, pred, prob):
    try:
        auc = float(roc_auc_score(y, prob, multi_class="ovr", average="macro", labels=[0,1,2,3]))
    except Exception:
        auc = None
    rec = recall_score(y, pred, labels=[0,1,2,3], average=None, zero_division=0)
    return {
        "accuracy": float(accuracy_score(y, pred)),
        "macro_f1": float(f1_score(y, pred, average="macro", zero_division=0)),
        "mcc": float(matthews_corrcoef(y, pred)),
        "roc_auc": auc,
        "recall_tier": {TIER_NAMES[t]: float(rec[t]) for t in range(4)},
    }

def train_one(idx, fusion, seed):
    set_seed(seed)
    Xt, Xe, Xtau, M, Y = to_t(MURIL_TR[idx], EMO_TR[idx], TEMP_TR[idx], MASK_TR[idx], y_train[idx])
    ds = TensorDataset(Xt, Xe, Xtau, M, Y.long())
    g = torch.Generator(); g.manual_seed(seed)
    dl = DataLoader(ds, batch_size=BATCH, shuffle=True, generator=g)
    counts = np.bincount(y_train[idx], minlength=4).astype(float)
    cls_w = torch.tensor(counts.sum() / (4.0 * np.clip(counts, 1, None)), dtype=torch.float, device=DEVICE)
    set_seed(seed)
    model = DriftCapsNet(MURIL_TR.shape[1], EMO_TR.shape[1], fusion=fusion).to(DEVICE)
    focal = FocalLoss(cls_w); opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    model.train()
    for ep in range(EPOCHS):
        for xt, xe, xtau, m, yb in dl:
            opt.zero_grad()
            logits, z = model(xt.to(DEVICE), xe.to(DEVICE), xtau.to(DEVICE), m.to(DEVICE), return_z=True)
            loss = focal(logits, yb.to(DEVICE)) + CONTRAST_W * supervised_contrastive(z, yb.to(DEVICE))
            loss.backward(); opt.step()
    return model

@torch.no_grad()
def evaluate(model, Xt, Xe, Xtau, M, y):
    model.eval()
    logits = model(torch.tensor(Xt).to(DEVICE), torch.tensor(Xe).to(DEVICE),
                   torch.tensor(Xtau).to(DEVICE), torch.tensor(M).to(DEVICE))
    prob = torch.softmax(logits, -1).cpu().numpy()
    return metric_suite(y, prob.argmax(1), prob)

log.info("drivers ready")

2026-08-08 20:26:54,867 | INFO | drivers ready


## 5. Multi-seed runs — full fused model + text-only ablation + baselines

In [6]:
from sklearn.linear_model import LogisticRegression

ALL_IDX = np.arange(len(train_rows))
METRICS = ["accuracy", "macro_f1", "mcc", "roc_auc"]

def majority_baseline(seed):
    maj = int(np.bincount(y_train, minlength=4).argmax())
    pred = np.full(len(y_test), maj)
    prob = np.zeros((len(y_test), 4)); prob[:, maj] = 1.0
    return metric_suite(y_test, pred, prob)

def probe_baseline(seed):       # MuRIL frozen-probe (LogReg) on cached embeddings (locked primary backbone, D2)
    clf = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=seed)
    clf.fit(MURIL_TR, y_train)
    pred = clf.predict(MURIL_TE)
    try:
        prob = clf.predict_proba(MURIL_TE)
    except Exception:
        prob = np.eye(4)[pred]
    return metric_suite(y_test, pred, prob)

per_seed = {"full": [], "text_only": [], "majority": [], "muril_probe": []}
t0 = time.time()
for s in SEEDS:
    m_full = evaluate(train_one(ALL_IDX, "attention", s), MURIL_TE, EMO_TE, TEMP_TE, MASK_TE, y_test)
    m_text = evaluate(train_one(ALL_IDX, "text_only", s), MURIL_TE, EMO_TE, TEMP_TE, MASK_TE, y_test)
    per_seed["full"].append(m_full); per_seed["text_only"].append(m_text)
    per_seed["majority"].append(majority_baseline(s)); per_seed["muril_probe"].append(probe_baseline(s))
    log.info("  seed %-4d | full macroF1=%.3f MCC=%.3f AUC=%s | text-only F1=%.3f | probe F1=%.3f",
             s, m_full["macro_f1"], m_full["mcc"],
             None if m_full["roc_auc"] is None else round(m_full["roc_auc"], 3),
             m_text["macro_f1"], per_seed["muril_probe"][-1]["macro_f1"])
log.info("multi-seed done in %.1fs", time.time() - t0)

2026-08-08 20:32:25,191 | INFO |   seed 42   | full macroF1=0.492 MCC=0.457 AUC=0.835 | text-only F1=0.366 | probe F1=0.234


2026-08-08 20:38:22,258 | INFO |   seed 2023 | full macroF1=0.475 MCC=0.452 AUC=0.83 | text-only F1=0.342 | probe F1=0.234


2026-08-08 20:45:06,691 | INFO |   seed 2024 | full macroF1=0.434 MCC=0.380 AUC=0.82 | text-only F1=0.410 | probe F1=0.234


2026-08-08 20:51:15,818 | INFO |   seed 0    | full macroF1=0.479 MCC=0.454 AUC=0.833 | text-only F1=0.365 | probe F1=0.234


2026-08-08 20:57:31,379 | INFO |   seed 7    | full macroF1=0.445 MCC=0.403 AUC=0.817 | text-only F1=0.365 | probe F1=0.234


2026-08-08 20:57:31,382 | INFO | multi-seed done in 1836.3s


## 6. Mean ± 95 % CI for every metric (t-distribution, df = n_seeds − 1)

In [7]:
def mean_ci(vals, conf=0.95):
    vals = [v for v in vals if v is not None]
    n = len(vals)
    if n == 0:
        return None
    m = float(np.mean(vals)); sd = float(np.std(vals, ddof=1)) if n > 1 else 0.0
    sem = sd / math.sqrt(n) if n > 1 else 0.0
    if HAS_SCIPY and n > 1:
        from scipy.stats import t as tdist
        h = float(tdist.ppf(0.5 + conf / 2, n - 1)) * sem
    else:
        h = 1.96 * sem
    return {"mean": round(m, 4), "sd": round(sd, 4), "ci95_lo": round(m - h, 4),
            "ci95_hi": round(m + h, 4), "half_width": round(h, 4), "n": n}

def collect(model_key, metric):
    return [d[metric] for d in per_seed[model_key]]

seed_metrics = {"part": "stage5_part5.1_statistical_robustness", "version": PART_VERSION,
                "principle": "P1 (system-computed); P3 (TEST scored once/seed, no selection on TEST);  multi-seed CI",
                "seeds": SEEDS, "models": {}}
for key in per_seed:
    seed_metrics["models"][key] = {
        "per_seed": per_seed[key],
        "ci": {mt: mean_ci(collect(key, mt)) for mt in METRICS},
        "recall_tier_ci": {TIER_NAMES[t]: mean_ci([d["recall_tier"][TIER_NAMES[t]] for d in per_seed[key]])
                           for t in range(4)},
    }

print("=== MEAN ± 95% CI (5 seeds) ===")
for key in ["full", "text_only", "muril_probe", "majority"]:
    ci = seed_metrics["models"][key]["ci"]
    print(f"  {key:12} macroF1={ci['macro_f1']['mean']:.3f} [{ci['macro_f1']['ci95_lo']:.3f},{ci['macro_f1']['ci95_hi']:.3f}]"
          f"  MCC={ci['mcc']['mean']:.3f} [{ci['mcc']['ci95_lo']:.3f},{ci['mcc']['ci95_hi']:.3f}]"
          f"  AUC={ci['roc_auc']['mean'] if ci['roc_auc'] else None}")

=== MEAN ± 95% CI (5 seeds) ===
  full         macroF1=0.465 [0.434,0.495]  MCC=0.429 [0.385,0.473]  AUC=0.8271
  text_only    macroF1=0.370 [0.339,0.400]  MCC=0.263 [0.221,0.305]  AUC=0.7345
  muril_probe  macroF1=0.234 [0.234,0.234]  MCC=0.007 [0.007,0.007]  AUC=0.472
  majority     macroF1=0.182 [0.182,0.182]  MCC=0.000 [0.000,0.000]  AUC=0.5


## 7. Significance — paired t-tests + Cohen's d (paired) + Benjamini–Hochberg FDR

Full fused model **vs each baseline**, paired across the 5 seeds, for **macro-F1** and **MCC**. **Cohen's d** and
**BH-FDR** are implemented from scratch; the FDR controls false discovery across the whole comparison family.

In [8]:
def paired_t(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float); diff = a - b
    if np.allclose(diff.std(ddof=1) if len(diff) > 1 else 0.0, 0.0):
        return (float("inf") if diff.mean() != 0 else 0.0), (0.0 if diff.mean() != 0 else 1.0)
    if HAS_SCIPY:
        from scipy.stats import ttest_rel
        r = ttest_rel(a, b)
        return float(r.statistic), float(r.pvalue)
    # fallback one-sample t on the differences
    n = len(diff); tval = diff.mean() / (diff.std(ddof=1) / math.sqrt(n))
    return float(tval), None

def cohens_d_paired(a, b):
    diff = np.asarray(a, float) - np.asarray(b, float)
    sd = diff.std(ddof=1)
    return float(diff.mean() / sd) if sd > 0 else (float("inf") if diff.mean() != 0 else 0.0)

def benjamini_hochberg(pvals, q=0.05):
    m = len(pvals); order = np.argsort(pvals); ranked = np.array(pvals)[order]
    thresh = [(i + 1) / m * q for i in range(m)]
    below = ranked <= thresh
    kmax = np.where(below)[0].max() + 1 if below.any() else 0
    reject = np.zeros(m, bool)
    if kmax > 0:
        reject[order[:kmax]] = True
    # BH-adjusted p-values (monotone)
    adj = np.empty(m); prev = 1.0
    for i in range(m - 1, -1, -1):
        prev = min(prev, ranked[i] * m / (i + 1)); adj[order[i]] = min(prev, 1.0)
    return reject.tolist(), [round(float(x), 4) for x in adj]

family, comparisons = [], []
for metric in ["macro_f1", "mcc"]:
    full_vals = collect("full", metric)
    for base in ["majority", "muril_probe", "text_only"]:
        base_vals = collect(base, metric)
        tval, p = paired_t(full_vals, base_vals)
        d = cohens_d_paired(full_vals, base_vals)
        comparisons.append({"metric": metric, "baseline": base,
                            "full_mean": round(float(np.mean(full_vals)), 4),
                            "base_mean": round(float(np.mean(base_vals)), 4),
                            "delta_mean": round(float(np.mean(full_vals) - np.mean(base_vals)), 4),
                            "t": round(tval, 4) if np.isfinite(tval) else None,
                            "p_raw": round(p, 5) if p is not None else None,
                            "cohens_d": round(d, 4) if np.isfinite(d) else None})
        family.append(p if p is not None else 1.0)

reject, p_adj = benjamini_hochberg(family, q=0.05)
for c, rj, pa in zip(comparisons, reject, p_adj):
    c["p_bh_fdr"] = pa; c["significant_q05"] = bool(rj)

significance = {"part": "stage5_part5.1_significance", "version": PART_VERSION,
                "method": "paired two-tailed t-test (df=4) + Cohen's d (paired) + Benjamini-Hochberg FDR q=0.05",
                "family_size": len(family), "comparisons": comparisons,
                "honest_note": "Pilot-scale, 5 seeds: low statistical power; some effects may not survive FDR "
                               "correction -> reported honestly (a non-significant result is a finding, not a "
                               "failure). CIs/tests bound SEED stability, not population generalization."}
print("=== SIGNIFICANCE (full vs baselines; paired, BH-FDR q=0.05) ===")
for c in comparisons:
    print(f"  {c['metric']:8} full-{c['baseline']:11} Δ={c['delta_mean']:+.3f} d={c['cohens_d']} "
          f"p_raw={c['p_raw']} p_FDR={c['p_bh_fdr']} sig={c['significant_q05']}")

=== SIGNIFICANCE (full vs baselines; paired, BH-FDR q=0.05) ===
  macro_f1 full-majority    Δ=+0.283 d=11.5932 p_raw=1e-05 p_FDR=0.0 sig=True
  macro_f1 full-muril_probe Δ=+0.231 d=9.4649 p_raw=3e-05 p_FDR=0.0 sig=True
  macro_f1 full-text_only   Δ=+0.095 d=2.118 p_raw=0.00906 p_FDR=0.0091 sig=True
  mcc      full-majority    Δ=+0.429 d=12.1496 p_raw=1e-05 p_FDR=0.0 sig=True
  mcc      full-muril_probe Δ=+0.422 d=11.9416 p_raw=1e-05 p_FDR=0.0 sig=True
  mcc      full-text_only   Δ=+0.166 d=3.5259 p_raw=0.0014 p_FDR=0.0017 sig=True


## 8. Stratified 5-fold on the YouTube TRAIN — internal stability

A TRAIN-internal cross-validation stability estimate (no TEST touched, P3). Folds are tiny at pilot scale → this
is a **stability check with logged N**, not a generalization claim. The drift case study (Parts 3.1 / 5.3) uses the
held-out Reddit/Twitter user histories as the **temporal holdout** — noted here, computed there.

In [9]:
from sklearn.model_selection import StratifiedKFold

KFOLD = 5
skf = StratifiedKFold(n_splits=KFOLD, shuffle=True, random_state=42)
fold_metrics = []
for fi, (tr_idx, va_idx) in enumerate(skf.split(ALL_IDX, y_train)):
    model = train_one(tr_idx, "attention", 42)
    m = evaluate(model, MURIL_TR[va_idx], EMO_TR[va_idx], TEMP_TR[va_idx], MASK_TR[va_idx], y_train[va_idx])
    fold_metrics.append(m)
    log.info("  fold %d/%d (train=%d val=%d): macroF1=%.3f MCC=%.3f",
             fi + 1, KFOLD, len(tr_idx), len(va_idx), m["macro_f1"], m["mcc"])

kfold = {"part": "stage5_part5.1_kfold", "version": PART_VERSION, "k": KFOLD,
         "scheme": "stratified k-fold on YouTube TRAIN (TRAIN-internal, no TEST leakage — P3)",
         "per_fold": fold_metrics,
         "ci": {mt: mean_ci([m[mt] for m in fold_metrics]) for mt in ["accuracy", "macro_f1", "mcc"]},
         "overfitting_guards": "dropout=0.3; frozen backbones; TRAIN-internal selection (no TEST in any loop).",
         "temporal_holdout_note": "The Reddit/Twitter user-history drift subset is the temporal holdout for the "
                                  "longitudinal Linguistic-Drift case study (Parts 3.1 / 5.3); never used in TRAIN.",
         "honest_note": "Pilot-scale: %d-fold over %d TRAIN rows -> folds are tiny (val ~%d); this is a stability "
                        "check with logged N, not a generalization estimate." % (KFOLD, len(train_rows),
                                                                                 len(train_rows)//KFOLD)}
kf = kfold["ci"]["macro_f1"]
print(f"=== k-fold ({KFOLD}-fold, TRAIN-internal) === macroF1 = {kf['mean']:.3f} "
      f"[{kf['ci95_lo']:.3f},{kf['ci95_hi']:.3f}]  MCC = {kfold['ci']['mcc']['mean']:.3f}")

2026-08-08 20:59:59,920 | INFO |   fold 1/5 (train=1576 val=395): macroF1=0.884 MCC=0.850


2026-08-08 21:02:43,166 | INFO |   fold 2/5 (train=1577 val=394): macroF1=0.790 MCC=0.756


2026-08-08 21:05:36,054 | INFO |   fold 3/5 (train=1577 val=394): macroF1=0.884 MCC=0.843


2026-08-08 21:08:26,754 | INFO |   fold 4/5 (train=1577 val=394): macroF1=0.799 MCC=0.757


2026-08-08 21:11:19,150 | INFO |   fold 5/5 (train=1577 val=394): macroF1=0.810 MCC=0.761


=== k-fold (5-fold, TRAIN-internal) === macroF1 = 0.834 [0.776,0.892]  MCC = 0.793


## 9. Persist seed_metrics + significance + kfold

In [10]:
(OUT_DIR / "seed_metrics.json").write_text(json.dumps(seed_metrics, indent=2, ensure_ascii=False), encoding="utf-8")
(OUT_DIR / "significance.json").write_text(json.dumps(significance, indent=2, ensure_ascii=False), encoding="utf-8")
(OUT_DIR / "kfold.json").write_text(json.dumps(kfold, indent=2, ensure_ascii=False), encoding="utf-8")
log.info("wrote seed_metrics.json, significance.json, kfold.json")

2026-08-08 21:11:19,249 | INFO | wrote seed_metrics.json, significance.json, kfold.json


## 10. Figure — multi-seed metric CIs (full vs baselines)

In [11]:
if HAS_MPL:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    keys = ["full", "text_only", "muril_probe", "majority"]
    labels = ["full (fused)", "text-only", "MuRIL probe", "majority"]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    for ax, metric, title in zip(axes, ["macro_f1", "mcc"], ["macro-F1", "MCC"]):
        means = [seed_metrics["models"][k]["ci"][metric]["mean"] for k in keys]
        errs  = [seed_metrics["models"][k]["ci"][metric]["half_width"] for k in keys]
        x = np.arange(len(keys))
        ax.bar(x, means, yerr=errs, capsize=5,
               color=["#1b7837", "#999999", "#7fbf7b", "#cccccc"])
        ax.set_xticks(x); ax.set_xticklabels(labels, rotation=20, ha="right", fontsize=8)
        ax.set_title("%s (mean ± 95%% CI, 5 seeds)" % title, fontsize=10)
        ax.axhline(0, color="k", lw=0.6)
    fig.tight_layout(); fig.savefig(OUT_DIR / "seed_ci.png", dpi=130); plt.close(fig)
    log.info("wrote seed_ci.png")
else:
    log.warning("matplotlib absent — skipped figure")

2026-08-08 21:11:21,742 | INFO | wrote seed_ci.png


## 11. This step is complete when

**notebook 14 is complete when:** the Objective 3 model is retrained under **≥5 seeds** with every metric as
**mean ± 95 % CI** (`seed_metrics.json`); **paired t-tests vs the three baselines + Cohen's d + Benjamini–Hochberg
FDR** are computed (`significance.json`); a **stratified k-fold** TRAIN-internal stability estimate exists
(`kfold.json`) with the temporal-holdout note + overfitting guards; `seed_ci.png` rendered; the **seed-stability ≠
population** caveat written. **No human; TEST scored once per seed, never used for selection.**

In [12]:
dod = all([
    (OUT_DIR / "seed_metrics.json").exists(),
    (OUT_DIR / "significance.json").exists(),
    (OUT_DIR / "kfold.json").exists(),
    len(per_seed["full"]) == len(SEEDS),
    all("p_bh_fdr" in c for c in comparisons),
])
fci = seed_metrics["models"]["full"]["ci"]
print("=" * 64); print("Objective 5 · notebook 14 — STATISTICAL ROBUSTNESS CARD"); print("=" * 64)
print("seeds          :", SEEDS)
print("full model     : macroF1 %.3f [%.3f,%.3f] | MCC %.3f [%.3f,%.3f] | AUC %s" % (
    fci["macro_f1"]["mean"], fci["macro_f1"]["ci95_lo"], fci["macro_f1"]["ci95_hi"],
    fci["mcc"]["mean"], fci["mcc"]["ci95_lo"], fci["mcc"]["ci95_hi"],
    fci["roc_auc"]["mean"] if fci["roc_auc"] else None))
nsig = sum(c["significant_q05"] for c in comparisons)
print("significance   : %d/%d comparisons survive BH-FDR q=0.05 (paired, 5 seeds)" % (nsig, len(comparisons)))
print("k-fold         : macroF1 %.3f (TRAIN-internal stability, %d-fold)" % (kfold["ci"]["macro_f1"]["mean"], KFOLD))
print("caveat         : CIs bound SEED stability, not population (pilot-scale)")
print("\nCompletion check:", "PASS" if dod else "INCOMPLETE", "(protocol validated; magnitudes provisional)")

Objective 5 · notebook 14 — STATISTICAL ROBUSTNESS CARD
seeds          : [42, 2023, 2024, 0, 7]
full model     : macroF1 0.465 [0.434,0.495] | MCC 0.429 [0.385,0.473] | AUC 0.8271
significance   : 6/6 comparisons survive BH-FDR q=0.05 (paired, 5 seeds)
k-fold         : macroF1 0.834 (TRAIN-internal stability, 5-fold)
caveat         : CIs bound SEED stability, not population (pilot-scale)

Completion check: PASS (protocol validated; magnitudes provisional)
